# NVIDIA Build Tool-Calling Detection (Wrench Icon)

This notebook verifies NVIDIA Build **tool-calling detection** via the **wrench icon** heuristic:

If the HTML contains any of:
- `wrench.svg`
- `data-icon-name="wrench"`
- `href="#wrench_..."`

then we treat the model as **tool-calling enabled** (per your rule).

It fetches the **base model page** (NOT `/modelcard`), e.g.:
- https://build.nvidia.com/openai/gpt-oss-120b


In [1]:
# %pip -q install httpx beautifulsoup4 lxml


In [1]:
import re, httpx

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://build.nvidia.com/",
}

WRENCH_PATTERNS = [
    re.compile(r"wrench\.svg", re.IGNORECASE),
    re.compile(r'data-icon-name\s*=\s*["\']wrench["\']', re.IGNORECASE),
    re.compile(r'href\s*=\s*["\']#wrench_\d+["\']', re.IGNORECASE),
    re.compile(r'id\s*=\s*["\']wrench_\d+["\']', re.IGNORECASE),
]

def has_wrench(html: str):
    html = html or ""
    hits = []
    for pat in WRENCH_PATTERNS:
        m = pat.search(html)
        if m:
            hits.append((pat.pattern, m.group(0)))
    return (len(hits) > 0), hits

def fetch(url: str):
    with httpx.Client(http2=True, follow_redirects=True, timeout=25.0) as client:
        r = client.get(url, headers=HEADERS)
        return r.status_code, r.text or ""


In [2]:
tests = [
    "https://build.nvidia.com/openai/gpt-oss-120b",
    "https://build.nvidia.com/deepseek-ai/deepseek-r1",
    "https://build.nvidia.com/deepseek-ai/deepseek-r1-0528",
]

for url in tests:
    code, html = fetch(url)
    ok, hits = has_wrench(html)
    print("="*120)
    print("URL:", url)
    print("HTTP:", code, "HTML_LEN:", len(html))
    print("WRENCH DETECTED:", ok)
    if hits:
        print("HITS:")
        for pat, sample in hits:
            print(" -", pat, "=>", sample)


URL: https://build.nvidia.com/openai/gpt-oss-120b
HTTP: 200 HTML_LEN: 482725
WRENCH DETECTED: False
URL: https://build.nvidia.com/deepseek-ai/deepseek-r1
HTTP: 200 HTML_LEN: 453580
WRENCH DETECTED: False
URL: https://build.nvidia.com/deepseek-ai/deepseek-r1-0528
HTTP: 200 HTML_LEN: 440981
WRENCH DETECTED: False


In [3]:
import httpx
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://build.nvidia.com/",
}

def fetch(url: str):
    with httpx.Client(http2=True, follow_redirects=True, timeout=25.0) as client:
        r = client.get(url, headers=HEADERS)
        return r.status_code, (r.text or "")

def has_tools_button(html: str):
    soup = BeautifulSoup(html or "", "lxml")

    for btn in soup.find_all("button"):
        txt = (btn.get_text(" ", strip=True) or "").strip()
        if txt != "Tools":
            continue

        attrs = btn.attrs or {}
        aria = str(attrs.get("aria-disabled", "")).strip().lower()
        data_state = str(attrs.get("data-state", "")).strip().lower()

        is_disabled = (
            ("disabled" in attrs) or
            ("data-disabled" in attrs) or
            (aria == "true") or
            (data_state == "disabled")
        )

        # evidence
        snippet = str(btn)[:700]
        return True, is_disabled, snippet, attrs

    return False, None, "", {}


In [4]:
tests = [
    "https://build.nvidia.com/openai/gpt-oss-120b",
    "https://build.nvidia.com/deepseek-ai/deepseek-r1",
    "https://build.nvidia.com/deepseek-ai/deepseek-r1-0528",
]

for url in tests:
    code, html = fetch(url)
    exists, disabled, snippet, attrs = has_tools_button(html)

    print("="*120)
    print("URL:", url)
    print("HTTP:", code, "HTML_LEN:", len(html))
    print("TOOLS BUTTON EXISTS:", exists)

    if exists:
        print("TOOLS BUTTON DISABLED:", disabled)
        print("ATTRS:", attrs)
        print("SNIPPET:", snippet)


URL: https://build.nvidia.com/openai/gpt-oss-120b
HTTP: 200 HTML_LEN: 482725
TOOLS BUTTON EXISTS: False
URL: https://build.nvidia.com/deepseek-ai/deepseek-r1
HTTP: 200 HTML_LEN: 453580
TOOLS BUTTON EXISTS: False
URL: https://build.nvidia.com/deepseek-ai/deepseek-r1-0528
HTTP: 200 HTML_LEN: 440981
TOOLS BUTTON EXISTS: False
